# 61 — RNA Pseudobulk DESeq2 (Cross-Species DE)

Runs after `scripts/create_rna_pseudobulk.py`.
Uses the same DESeq2 methodology as the ATAC pipeline (Section 11 of deseq2_utils.R).

In [1]:
# Cell 1: Packages & utilities
suppressPackageStartupMessages({
  library(DESeq2); library(arrow)
  library(ggplot2); library(dplyr); library(tibble); library(readr)
})
source("../src/deseq2_utils.R")
message("Packages loaded.")

Packages loaded.



In [2]:
# Cell 2: Config
BASE_RNA <- "/links/groups/treutlein/USERS/jjans/analysis/adult_intestine/rna"
PB_DIR   <- file.path(BASE_RNA, "pseudobulk_deseq2")
OUT_DIR  <- file.path(BASE_RNA, "pseudobulk_deseq2")   # results saved here too
SPECIES  <- c("Human", "Bonobo", "Chimpanzee", "Gorilla", "Macaque", "Marmoset")

# QC thresholds (RNA is much deeper than ATAC regions; use counts not cells)
MIN_COUNTS <- 500
MIN_CELLS  <- 10

# Check pseudobulk files exist
for (sp in SPECIES) {
  f <- file.path(PB_DIR, sp, "pseudobulk_counts.parquet")
  if (!file.exists(f)) stop("Missing pseudobulk for ", sp, ": run create_rna_pseudobulk.py first")
}
message("All pseudobulk files found.")

All pseudobulk files found.



In [3]:
# Cell 3: Load & merge RNA pseudobulk data
raw_rna <- load_rna_pseudobulk(PB_DIR, SPECIES)

# Inner join: shared genes across all species
merged_rna <- merge_rna_pseudobulk(raw_rna$all_counts, raw_rna$all_meta, join_type = "inner")
counts_rna <- merged_rna$counts
meta_rna   <- merged_rna$meta

message("Shared genes: ", nrow(counts_rna))
message("Total samples: ", ncol(counts_rna))
print(table(meta_rna$species, meta_rna$cell_type))

Loaded Human: 12785 genes x 227 samples



Loaded Bonobo: 12785 genes x 103 samples



Loaded Chimpanzee: 12785 genes x 104 samples



Loaded Gorilla: 12785 genes x 166 samples



Loaded Macaque: 12785 genes x 81 samples



...

Inner join: 12785 shared genes



Merged RNA matrix: 12785 genes x 795 samples



Shared genes: 12785



Total samples: 795



            
             Adipocytes BEST4+ cells Colonocytes Crypt Fibroblasts WNT2B+ ECs
  Bonobo              4            2           1                        4   4
  Chimpanzee          2            1           2                        4   4
  Gorilla             1            1           2                        6   7
  Human               2           10           6                        9  10
  Macaque             3            3           2                        3   3
  Marmoset            2            2           2                        4   4
            
             EECs Enteric glia Enteric neurons Enterocytes Eosinophils
  Bonobo        3            4               0           2           1
  Chimpanzee    4            4               0           3           0
  Gorilla       5            7               0           3           6
  Human        10            7               2           4           0
  Macaque       3            3               1           1           0
  

In [4]:
# Cell 4: Aggregate across regions (Individual x cell_type, summing Duodenum + Colon)
agg <- aggregate_rna_pseudobulk(counts_rna, meta_rna)
counts_agg <- agg$counts
meta_agg   <- agg$meta

message("After region aggregation: ", ncol(counts_agg), " samples")
message("Samples per species:")
print(table(meta_agg$species))
print(table(meta_agg$cell_type))

After region aggregation: 672 samples



Samples per species:




    Bonobo Chimpanzee    Gorilla      Human    Macaque   Marmoset 
       103        104        166        151         60         88 



                         Adipocytes                        BEST4+ cells 
                                 12                                  14 
                        Colonocytes            Crypt Fibroblasts WNT2B+ 
                                 15                                  24 
                                ECs                                EECs 
                                 26                                  22 
                       Enteric glia                     Enteric neurons 
                                 25                                   6 
                        Enterocytes                         Eosinophils 
                                 15                                   7 
                       Goblet cells                                ICCs 
                                 26                                  16 
                               ILCs                       Lymphatic ECs 
                                  2               

In [5]:
# Cell 5: Sample QC — remove low-coverage pseudobulks
meta_agg$total_counts <- colSums(counts_agg)
keep <- meta_agg$total_counts >= MIN_COUNTS & meta_agg$n_cells >= MIN_CELLS
message("Samples passing QC (counts>=", MIN_COUNTS, ", cells>=", MIN_CELLS, "): ",
        sum(keep), "/", ncol(counts_agg))
counts_agg <- counts_agg[, keep, drop = FALSE]
meta_agg   <- meta_agg[keep, , drop = FALSE]

# Factor setup
meta_agg$species   <- factor(meta_agg$species, levels = SPECIES)
meta_agg$cell_type <- factor(make.names(as.character(meta_agg$cell_type)))
meta_agg$donor     <- factor(meta_agg$donor)

# Save QC-filtered pseudobulk RDS for downstream notebooks
saveRDS(list(counts = counts_agg, meta = meta_agg),
        file.path(PB_DIR, "rna_pb_data_aggregated.rds"))
message("Saved rna_pb_data_aggregated.rds")

Samples passing QC (counts>=500, cells>=10): 672/672



Saved rna_pb_data_aggregated.rds



In [6]:
# Cell 6: PCA sanity check
vst_rna <- vst(DESeqDataSetFromMatrix(
  countData = counts_agg,
  colData   = meta_agg,
  design    = ~ species
), blind = TRUE)

pca_df <- as.data.frame(prcomp(t(assay(vst_rna)), scale. = FALSE)$x[, 1:3])
pca_df <- cbind(pca_df, meta_agg)

p <- ggplot(pca_df, aes(PC1, PC2, color = species, shape = cell_type)) +
  geom_point(size = 2, alpha = 0.8) +
  theme_bw() + ggtitle("RNA pseudobulk PCA (VST, shared genes)")
ggsave(file.path(PB_DIR, "RNA_PCA_species.pdf"), p, width = 9, height = 6)
message("PCA plot saved.")

Warning message:
"The shape palette can deal with a maximum of 6 discrete values because more
than 6 becomes difficult to discriminate
i you have requested 40 values. Consider specifying shapes manually if you need
  that many of them."


Warning message:
"Removed 559 rows containing missing values or values outside the scale range
(`geom_point()`)."


PCA plot saved.



In [7]:
# Cell 7: DESeq2 — Species-specific (each species vs rest, like shared-peak ATAC)
message("\n========== Species-specific RNA DE ==========")
rna_sp_res <- run_deseq2_rna_species(
  counts_rna   = counts_agg,
  meta_rna     = meta_agg,
  species      = SPECIES,
  out_dir      = OUT_DIR,
  min_samples  = 2,
  filter_outliers = TRUE
)
message("Species-specific RNA DE done. Saved to rna_differential_results/species_specific/")


========== Species-specific RNA DE ==========




=== [RNA species] TA.cells (26 samples) ===



  Human       : 1140 UP (padj<0.05, LFC>1)



  Bonobo      : 202 UP (padj<0.05, LFC>1)



  Chimpanzee  : 86 UP (padj<0.05, LFC>1)



...


=== [RNA species] Specialized.Fibroblasts.ADAMTS16. (1 samples) ===




=== [RNA species] Specialized.Fibroblasts.PCDH9. (1 samples) ===




=== [RNA species] RBP2_high.cells (1 samples) ===



RNA species DE complete.



Species-specific RNA DE done. Saved to rna_differential_results/species_specific/



In [8]:
# Cell 8: DESeq2 — Evolutionary contrasts (same 19 contrasts as ATAC)
message("\n========== Evolutionary RNA DE ==========")
evo_contrasts <- define_evolutionary_contrasts()
message("Running ", length(evo_contrasts), " contrasts")

rna_evo_res <- run_deseq2_rna_evolutionary(
  counts_rna      = counts_agg,
  meta_rna        = meta_agg,
  evo_contrasts   = evo_contrasts,
  out_dir         = OUT_DIR,
  min_samples     = 2,
  filter_outliers = TRUE
)
message("Evolutionary RNA DE done. Saved RNA_DE_res_list.rds")


========== Evolutionary RNA DE ==========



Defined 19 evolutionary contrasts.



Running 19 contrasts




=== [RNA] TA.cells ===



  Node1_Human_vs_Pan                      : 1923 UP, 1141 DOWN  (padj<0.05, |LFC|>1)



...


=== [RNA] Specialized.Fibroblasts.ADAMTS16. ===




=== [RNA] Specialized.Fibroblasts.PCDH9. ===




=== [RNA] RBP2_high.cells ===




RNA evolutionary DE complete. Saved RNA_DE_res_list.rds



Evolutionary RNA DE done. Saved RNA_DE_res_list.rds



In [9]:
# Cell 9: Quick summary — how many DE genes per cell type × contrast?
summary_rows <- list()
for (ct in names(rna_evo_res)) {
  for (cn in names(rna_evo_res[[ct]])) {
    res <- as.data.frame(rna_evo_res[[ct]][[cn]])
    n_up   <- sum(!is.na(res$padj) & res$padj < 0.05 & res$log2FoldChange > 1)
    n_down <- sum(!is.na(res$padj) & res$padj < 0.05 & res$log2FoldChange < -1)
    summary_rows[[paste(ct, cn)]] <- data.frame(
      cell_type = ct, contrast = cn, n_up = n_up, n_down = n_down
    )
  }
}
de_summary <- do.call(rbind, summary_rows)
write.csv(de_summary, file.path(OUT_DIR, "rna_differential_results", "RNA_DE_summary.csv"),
          row.names = FALSE)
message("\nDE gene counts per cell type × contrast:")
print(de_summary[order(de_summary$n_up, decreasing = TRUE), ][1:30,])


DE gene counts per cell type <U+00D7> contrast:



                                                                                     cell_type
TA.cells Pair_Human_vs_Marmoset                                                       TA.cells
Colonocytes Pair_Human_vs_Marmoset                                                 Colonocytes
Goblet.cells Pair_Human_vs_Marmoset                                               Goblet.cells
TA.cells Pair_Human_vs_Macaque                                                        TA.cells
Enterocytes Pair_Human_vs_Marmoset                                                 Enterocytes
TA.cells Node4_OldWorld_vs_Marmoset                                                   TA.cells
Colonocytes Pair_Human_vs_Macaque                                                  Colonocytes
Stem.cells Pair_Human_vs_Marmoset                                                   Stem.cells
Goblet.cells Pair_Human_vs_Macaque                                                Goblet.cells
Crypt.Fibroblasts.WNT2B. Pair_Human_vs_Marmoset   

In [10]:
# Cell 10: Final checkpoint
message("\n=== RNA DE notebook complete ===")
message("Output: ", file.path(OUT_DIR, "rna_differential_results"))
message("  RNA_DE_res_list.rds                (evolutionary contrasts)")
message("  species_specific/RNA_DE_species_res_list.rds")
message("  RNA_DE_summary.csv")


=== RNA DE notebook complete ===



Output: /links/groups/treutlein/USERS/jjans/analysis/adult_intestine/rna/pseudobulk_deseq2/rna_differential_results



  RNA_DE_res_list.rds                (evolutionary contrasts)



  species_specific/RNA_DE_species_res_list.rds



  RNA_DE_summary.csv

